In [1]:
!hostname

gpue03.delta.ncsa.illinois.edu


In [3]:
# in a fresh notebook cell, BEFORE any other import
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
print(jax.devices())                       # should print [CpuDevice(...)]

import pickle
with open("/projects/bhdw/asachan/tmp/tp_atac_rna_skm.pkl", "rb") as f:
    tp = pickle.load(f)


[CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]


In [4]:
import sys
print(sys.executable)

/projects/bhdw/asachan/.conda/envs/moscot/bin/python


In [5]:
import scanpy as sc
import scipy.sparse as sp
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
ad.settings.allow_write_nullable_strings = True
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
import os
os.chdir('/projects/bgdb/asachan/methods/FIREFate/moscot')  # directory containing utils.py
import sys
import logging
import warnings

export_dir = "/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human"
human_genome_path = '/work/hdd/bgdb/asachan/datasets_proj/human_genome_files'

out_tmp = '/projects/bhdw/asachan/tmp'

In [7]:
pd.set_option('mode.string_storage', 'python')

In [8]:
import moscot

### Load learnt coupling matrix between atac and rna and joint embeddings

In [9]:
joint = ad.read_h5ad('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/joint_atac_rna_female_type2.h5ad')

In [10]:
adata_atac = sc.read_h5ad('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/atac_objects/atac_fiber/atac_female_type2.h5ad')

In [11]:
# add age as categorical
adata_atac.obs["age_categorical"] = adata_atac.obs["age"].astype("category")

In [12]:
adata_rna = sc.read_h5ad('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/rna_objects/rna_female_type2_ds_wrt_HALLMARK_DNA_REPAIR.h5ad')

#### Pull soft peak accessibility onto every RNA cell - controlled by age so that only old peaks are mapped to old rna and same for young

In [13]:
import numpy as np, scipy.sparse as sp, jax

# ---- per-sample T as numpy (CPU-only path through jax) -----------------
cpu = jax.devices("cpu")[0]
def transport_numpy(prob):
    o   = prob.solution._output
    T_j = jax.device_put(o.linear_state.matrix,           cpu)
    old = jax.device_put(o.old_transport_mass,            cpu)
    new = jax.device_put(o.linear_state.transport_mass,   cpu)
    return np.asarray(T_j) * float(np.sqrt(np.asarray(old) / np.asarray(new)))

# sample → age (verify these against your metadata!)
sample_to_age = (
    adata_atac.obs[["sample", "age_categorical"]]
      .drop_duplicates().set_index("sample")["age_categorical"]
      .astype(str).to_dict()
)
print("sample → age:", sample_to_age)

# ---- stack per-sample T's, tag rows by age -----------------------------
T_blocks, atac_obs_idx, atac_age_blocks = [], [], []
for (src_b, tgt_b), prob in tp.problems.items():
    T_blocks.append(transport_numpy(prob))                    # (n_atac_b, n_rna)
    a_idx = adata_atac.obs.index[adata_atac.obs["sample"] == src_b]
    atac_obs_idx.append(a_idx)
    atac_age_blocks.append(np.full(len(a_idx), sample_to_age[src_b]))

T_full   = np.vstack(T_blocks)
atac_idx = np.concatenate(atac_obs_idx)
atac_age = np.concatenate(atac_age_blocks)
rna_age  = adata_rna.obs["age_categorical"].astype(str).values

# ---- AGE-MATCHED stratification ---------------------------------------
T_strat = T_full.copy()
for a in np.unique(atac_age):
    cross = atac_age != a
    cols  = rna_age == a
    if cross.any() and cols.any():
        T_strat[np.ix_(cross, cols)] = 0

col_sum = T_strat.sum(0, keepdims=True)
zeroed  = (col_sum.ravel() == 0).sum()
print(f"RNA cells with no age-matched ATAC source: {zeroed} / {T_strat.shape[1]}")
T_col   = T_strat / np.where(col_sum > 0, col_sum, 1.0)

# ---- soft peak matrix per RNA cell (raw counts, not TF-IDF) -----------
X_peak = adata_atac[atac_idx].X
X_peak = X_peak.toarray() if sp.issparse(X_peak) else X_peak
X_peaks_rna = T_col.T @ X_peak                                # (n_rna, n_peaks)

adata_rna.obsm["X_peaks_inferred"] = X_peaks_rna.astype(np.float32)
print("inferred peaks per RNA cell:", X_peaks_rna.shape,
      "  mean nnz / cell:", float((X_peaks_rna > 0).sum(1).mean()))

sample → age: {'YM2': '34', 'OM6': '80', 'OM9': '80'}


E0501 09:20:24.896432  206104 cuda_dnn.cc:466] Could not create cudnn handle: CUDNN_STATUS_NOT_INITIALIZED
E0501 09:20:24.896484  206104 cuda_dnn.cc:470] Memory usage: 36966432768 bytes free, 150121283584 bytes total.
E0501 09:20:24.896512  206104 cuda_dnn.cc:480] Possibly insufficient driver version: 570.148.8
E0501 09:20:25.253030  206104 cuda_dnn.cc:466] Could not create cudnn handle: CUDNN_STATUS_NOT_INITIALIZED
E0501 09:20:25.253068  206104 cuda_dnn.cc:470] Memory usage: 36966432768 bytes free, 150121283584 bytes total.
E0501 09:20:25.253089  206104 cuda_dnn.cc:480] Possibly insufficient driver version: 570.148.8


JaxRuntimeError: FAILED_PRECONDITION: DNN library initialization failed. Look at the errors above for more details.

In [ ]:
import pandas as pd
print(pd.DataFrame({
    "rna_age": rna_age,
    "received_mass": col_sum.ravel(),
}).groupby("rna_age")["received_mass"].agg(["mean", "std", "min", "max", "count"]))

#### is the transition probability too diffused or does it still capture geometrical structure above uniform coupling?
#### also fewer atac-neighbours

In [ ]:
import numpy as np, pandas as pd
T_col_n = T_strat / np.where(col_sum > 0, col_sum, 1.0)
H       = -(T_col_n * np.log(T_col_n + 1e-30)).sum(0)
df = pd.DataFrame({
    "age":    rna_age,
    "top1":   T_col_n.max(0),
    "eff_k":  np.exp(H),
})
print(df.groupby("age").agg(["median", lambda x: np.percentile(x, 95)]))

#### Multi-ome data

In [ ]:
import muon as mu
# Import a module with ATAC-seq-related functions
from muon import atac as ac

In [ ]:
adata_atac

In [ ]:
# %% TF-IDF normalisation
# muon's tfidf writes the normalised matrix back into .X
ac.pp.tfidf(adata_atac, scale_factor=1e4)

In [ ]:
#add it to layer 
adata_atac.layers["tfidf"] = adata_atac.X.copy()

In [ ]:
# ---- (cell 20 replacement) build MuData from raw soft peaks -----------
import mudata as mu, anndata as ad

peaks    = adata_atac.var_names.to_list()
peak_var = adata_atac.var[["chrom","start","end"]].copy() \
           if {"chrom","start","end"}.issubset(adata_atac.var.columns) else None
if peak_var is None:
    parsed = [p.replace(":", "-").split("-") for p in peaks]
    peak_var = pd.DataFrame(parsed, columns=["chrom","start","end"], index=peaks) \
                 .astype({"start": int, "end": int})

rna  = ad.AnnData(
    X=adata_rna.X if sp.issparse(adata_rna.X) else sp.csr_matrix(adata_rna.X),
    obs=adata_rna.obs.copy(), var=adata_rna.var.copy(),
    obsm={"X_pca": adata_rna.obsm["X_pca"]},
)
atac = ad.AnnData(
    X=sp.csr_matrix(X_peaks_rna),
    obs=adata_rna.obs.copy(), var=peak_var,
)
atac.var_names = peaks
mome = mu.MuData({"rna": rna, "atac": atac})

# ---- (cell 21 replacement) TF-IDF in-place on mome ATAC ---------------
def tfidf(M, scale=1e4):
    M = M.tocsr() if sp.issparse(M) else sp.csr_matrix(M)
    rs  = np.asarray(M.sum(1)).ravel() + 1e-12
    cs  = np.asarray(M.sum(0)).ravel() + 1e-12
    idf = np.log1p(M.shape[0] / cs)
    Mn  = sp.diags(scale / rs) @ M @ sp.diags(idf)
    return Mn.log1p()

mome["atac"].layers["tfidf"] = tfidf(mome["atac"].X)
print(mome)

#### Subset the rna to MaxToki vocab

In [ ]:
mome["rna"].var_names

In [ ]:
import pandas as pd, pyranges as pr, numpy as np

# 1. symbol → ENSG from the same GTF you used for TSS lookup
gtf = pr.read_gtf(f"{human_genome_path}/gencode.v46.annotation.gtf").df
gtf = gtf[gtf.Feature == "gene"][["gene_name", "gene_id"]].drop_duplicates("gene_name")
gtf["gene_id_clean"] = gtf["gene_id"].str.split(".").str[0]    # strip ENSG version
sym2ensg = dict(zip(gtf["gene_name"].astype(str), gtf["gene_id_clean"]))
print(f"GTF mappings: {len(sym2ensg)} symbol→ENSG")

# 2. attach ENSG to mome["rna"].var
syms = mome["rna"].var_names.astype(str)
mome["rna"].var["ensg"] = [sym2ensg.get(s, None) for s in syms]

n_total   = mome["rna"].n_vars
n_mapped  = mome["rna"].var["ensg"].notna().sum()
print(f"{n_mapped}/{n_total} symbols mapped to ENSG  ({n_mapped/n_total:.1%})")

In [ ]:
import json, numpy as np, pandas as pd

# 1. load MaxToki vocab
with open("/projects/bhdw/asachan/methods/maxtoki-perturb/src/maxtoki_mlx/resources/token_dictionary.json") as f:
    vocab = json.load(f)

# strip special tokens to leave only ENSG entries
ensg_to_token = {k: v for k, v in vocab.items() if k.startswith("ENSG")}
print(f"vocab tokens: {len(vocab)} total, {len(ensg_to_token)} ENSG-keyed")

# 2. attach token index to mome["rna"].var
mome["rna"].var["maxtoki_token"] = (
    mome["rna"].var["ensg"].map(ensg_to_token).astype("Int64")  # nullable int
)

n_in_vocab = mome["rna"].var["maxtoki_token"].notna().sum()
print(f"vars with MaxToki token: {n_in_vocab} / {mome['rna'].n_vars} "
      f"({n_in_vocab / mome['rna'].n_vars:.1%})")

In [ ]:
import numpy as np

mask = mome["rna"].var["maxtoki_token"].notna()
print(f"keeping {mask.sum()} / {mome['rna'].n_vars} genes")

# rebuild MuData with subsetted RNA modality, ATAC unchanged
import mudata as mu
rna_sub = mome["rna"][:, mask.values].copy()
rna_sub.var["maxtoki_token"] = rna_sub.var["maxtoki_token"].astype(int)
rna_sub = rna_sub[:, np.argsort(rna_sub.var["maxtoki_token"].values)].copy()  # sort by token id


In [ ]:
# After your subsetting block, before saving
original_rna = mome["rna"]   # whatever the pre-mask version is

# Match by gene id (since order may differ)
gene_to_idx = {g: i for i, g in enumerate(original_rna.var["ensembl_id"].astype(str))}
src_idx = [gene_to_idx[g] for g in rna_sub.var["ensembl_id"].astype(str)]

# Port counts (and any other layer you need)
for layer_name in ("counts",):    # add "lognorm", "spliced", etc. if needed
    if layer_name in original_rna.layers:
        rna_sub.layers[layer_name] = original_rna.layers[layer_name][:, src_idx].copy()
        print(f"[port] {layer_name}: {rna_sub.layers[layer_name].shape}")

mome = mu.MuData({"rna": rna_sub, "atac": mome["atac"]})
print(mome)

In [ ]:
#overwrite mome file 
mome.write_h5mu(f"{out_tmp}/mome_atac_rna_skm.h5mu")

In [ ]:
rna_sub.var["ensembl_id"] = rna_sub.var['ensg']        # if ENSGs are the index

In [ ]:
# write the rna view of this mudata file 
rna_sub.write_h5ad(f"{out_tmp}/rna_view_muon.h5ad")

In [ ]:
rna_maxtoki_input = sc.read_h5ad(f"{out_tmp}/rna_view_muon.h5ad")

In [ ]:
rna_maxtoki_input.var["ensembl_id"]

# Peak to gene (TSS +- 250kb for long-range CRE-peak capture & TF-Motif scan in age-specific open chromatin)

#### TSS finder

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, pyranges as pr
from tqdm.auto import tqdm

# TSS lookup
gtf = pr.read_gtf(f"{human_genome_path}/gencode.v46.annotation.gtf").df
gtf = gtf[gtf.Feature == "gene"][["gene_id","Chromosome","Start","End","Strand"]]
gtf["TSS"]     = np.where(gtf.Strand == "+", gtf.Start, gtf.End)
gtf["gene_id"] = gtf["gene_id"].str.split(".").str[0]
tss = gtf.drop_duplicates("gene_id").set_index("gene_id")[["Chromosome","TSS"]]

atac, rna = mome["atac"], mome["rna"]
genes_e   = rna.var["ensg"].astype(str).values
peak_mid  = ((atac.var["start"].astype(int) + atac.var["end"].astype(int)) // 2).values
peak_chr  = atac.var["chrom"].astype(str).values
peak_by_chr = {ch: (np.where(peak_chr==ch)[0], peak_mid[peak_chr==ch])
               for ch in np.unique(peak_chr)}

WIN, SCALE = 250_000, 50_000 # distance -decay based likelihood
present = np.isin(genes_e, tss.index.values)
print(f"{present.sum():,}/{len(genes_e):,} genes have TSS")

cand_p, cand_g, cand_w = [], [], []
for gi in tqdm(np.where(present)[0], desc="P2G distance-decay"):
    e = genes_e[gi]
    chrom, t = tss.at[e, "Chromosome"], int(tss.at[e, "TSS"])
    if chrom not in peak_by_chr: continue
    p_idx, p_mid = peak_by_chr[chrom]
    d = np.abs(p_mid - t)
    m = d <= WIN
    if m.any():
        cand_p.append(p_idx[m])
        cand_g.append(np.full(m.sum(), gi, dtype=np.int64))
        cand_w.append(np.exp(-d[m] / SCALE).astype(np.float32))

cand_p = np.concatenate(cand_p); cand_g = np.concatenate(cand_g); cand_w = np.concatenate(cand_w)
P2G = sp.csr_matrix((cand_w, (cand_p, cand_g)), shape=(atac.n_vars, rna.n_vars))
print("P2G:", P2G.shape, "nnz:", P2G.nnz)
sp.save_npz(f"{out_tmp}/P2G.npz", P2G)

In [ ]:
import numpy as np, scipy.sparse as sp, pandas as pd
import matplotlib.pyplot as plt

X_open = mome["atac"].X.toarray() if sp.issparse(mome["atac"].X) else mome["atac"].X
# (n_cells, n_peaks) @ (n_peaks, n_genes) = (n_cells, n_genes) regulatory mass
reg_mass = X_open @ P2G                                                      # dense

ages = mome["rna"].obs["age_categorical"].astype(str).values
mask80 = ages == "80"; mask34 = ages == "34"

mean80 = reg_mass[mask80].mean(0); mean34 = reg_mass[mask34].mean(0)
log2fc = np.log2((mean80 + 1e-6) / (mean34 + 1e-6))

# rough significance via Welch t-test
from scipy.stats import ttest_ind
_, pvals = ttest_ind(reg_mass[mask80], reg_mass[mask34], axis=0, equal_var=False)
nlogp = -np.log10(np.clip(pvals, 1e-50, 1))


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

hits = (np.abs(log2fc) > 0.5) & (pvals < 1e-5)

N = 5
nlogp_safe = np.where(hits, nlogp, -np.inf)
top_up   = np.argsort(np.where(log2fc > 0, nlogp_safe, -np.inf))[::-1][:N]
top_down = np.argsort(np.where(log2fc < 0, nlogp_safe, -np.inf))[::-1][:N]

top_df = pd.DataFrame({
    "gene":      np.concatenate([mome["rna"].var_names.values[top_up],
                                 mome["rna"].var_names.values[top_down]]),
    "log2fc":    np.concatenate([log2fc[top_up],   log2fc[top_down]]),
    "-log10(p)": np.concatenate([nlogp[top_up],    nlogp[top_down]]),
    "side":      ["80↑"] * N + ["34↑"] * N,
})
print(top_df)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(log2fc, nlogp, s=4, alpha=0.4, c="grey")
ax.scatter(log2fc[hits], nlogp[hits], s=8, c="crimson")
ax.axhline(-np.log10(1e-5), ls="--", c="k", lw=0.6)
ax.axvline( 0.5, ls="--", c="k", lw=0.6); ax.axvline(-0.5, ls="--", c="k", lw=0.6)
ax.set_xlabel("log2( mass[80] / mass[34] )")
ax.set_ylabel("-log10(p)")
ax.set_title("Differential per-gene peak mass transported through GW-OT: 80yo vs 34yo")

# legend-style text boxes inside the plot
def gene_box(genes, color, header, loc):
    text = header + "\n" + "\n".join(genes)
    box  = AnchoredText(
        text, loc=loc, prop=dict(color=color, fontsize=10, fontweight="bold"),
        frameon=True, borderpad=0.5, pad=0.4,
    )
    box.patch.set_boxstyle("round,pad=0.3")
    box.patch.set_edgecolor(color)
    box.patch.set_facecolor("white")
    box.patch.set_alpha(0.85)
    return box

ax.add_artist(gene_box(
    [mome["rna"].var_names[gi] for gi in top_down],
    color="darkred",  header="More open TSS sites in young (34 y/o)", loc="upper left"
))
ax.add_artist(gene_box(
    [mome["rna"].var_names[gi] for gi in top_up],
    color="darkblue", header="More open TSS sites in old (80 y/o)",  loc="upper right"
))

plt.tight_layout()
fig.savefig(f"{out_tmp}/volcano_p2g_age.svg", format="svg", bbox_inches="tight")
plt.show()

### TF Motif Scan per peak

In [ ]:
import os, numpy as np, scipy.sparse as sp, pyfaidx
from memelite import fimo
from memelite.io import read_meme
from tqdm.auto import tqdm

# write peaks to FASTA with integer-indexed headers
fa_path = f"{out_tmp}/peaks.fa"
fa = pyfaidx.Fasta(f"{human_genome_path}/hg38.fa", as_raw=True)
with open(fa_path, "w") as f:
    for i, (_, r) in enumerate(tqdm(mome["atac"].var.iterrows(),
                                    total=mome["atac"].n_vars, desc="write peaks.fa")):
        seq = str(fa[r["chrom"]][int(r["start"]):int(r["end"])])
        f.write(f">{i}\n{seq}\n")

# load JASPAR vertebrate motifs
motifs      = read_meme(f"{human_genome_path}/JASPAR2024_CORE_vertebrates.meme")
motif_names = np.array(list(motifs.keys()))
print(f"{len(motif_names)} motifs to scan")

# run FIMO (numba-parallel across motifs; ~3-10 min on 48 CPUs)
hits = fimo(motifs, fa_path, threshold=1e-4, reverse_complement=True, dim=0)
print(f"{sum(len(df) for df in hits):,} total motif hits")

In [ ]:
rows, cols = [], []
for m_idx, df in enumerate(tqdm(hits, desc="build MOTIF")):
    if len(df) == 0: continue
    rows.extend(df["sequence_name"].astype(int).tolist())
    cols.extend([m_idx] * len(df))

MOTIF = sp.csr_matrix(
    (np.ones(len(rows), dtype=np.uint8), (rows, cols)),
    shape=(mome["atac"].n_vars, len(motif_names))
)
print("MOTIF:", MOTIF.shape, "nnz:", MOTIF.nnz)

# sanity stats
n_per_motif = (MOTIF > 0).sum(0).A1
n_per_peak  = (MOTIF > 0).sum(1).A1
print(f"peaks with ≥1 hit : {(n_per_peak > 0).sum()} / {MOTIF.shape[0]}")
print(f"  median peaks/motif: {np.median(n_per_motif):.0f}")
print(f"  median motifs/peak: {np.median(n_per_peak):.0f}")

sp.save_npz(f"{out_tmp}/MOTIF.npz", MOTIF)
np.save     (f"{out_tmp}/motif_names.npy", motif_names)

In [ ]:
print("first motif_names:", motif_names[:8])

# Build bf16 attention bias

In [ ]:
import json, pandas as pd

# 1. chromatin base GRN (CellOracle-style — full, no expression filter)
G_base = (MOTIF.T @ P2G).tocsr()
sp.save_npz(f"{out_tmp}/G_base.npz", G_base)
print("G_base:", G_base.shape, "nnz:", G_base.nnz)

# 2. motif → vocab token (deferred until attention time)
sym2ensg_u = {k.upper(): v for k, v in sym2ensg.items()}        # from cell 25

with open("/projects/bhdw/asachan/methods/maxtoki-perturb/src/maxtoki_mlx/resources/token_dictionary.json") as f:
    vocab = json.load(f)
ensg_to_token = {k: v for k, v in vocab.items() if k.startswith("ENSG")}

motif_to_sym    = pd.Series(motif_names).str.split("_").str[0].str.upper()
motif_to_ensg   = motif_to_sym.map(sym2ensg_u)
motif_token_arr = motif_to_ensg.map(ensg_to_token).fillna(-1).astype(np.int64).values
gene_token_arr  = mome["rna"].var["maxtoki_token"].astype(int).values

print(f"motifs total              : {len(motif_names)}")
print(f"  → ENSG resolved         : {motif_to_ensg.notna().sum()}")
print(f"  → MaxToki token         : {(motif_token_arr >= 0).sum()}")
print(f"  unique TF tokens        : {pd.Series(motif_token_arr[motif_token_arr>=0]).nunique()}")

np.save(f"{out_tmp}/motif_token_arr.npy", motif_token_arr)
np.save(f"{out_tmp}/gene_token_arr.npy",  gene_token_arr)